# 07 — GNN graph generation

This notebook starts the G7 graph-generation pipeline. T9.1 only loads the AqSolDB records, the persisted 3D conformers, and the complete `GNNConfig`. Graph construction and display are intentionally deferred to later tasks.

The conformer artifact is expected at `data/conformers/aqsoldb/conformers.pkl`. Each record must contain `mol_id`, `smiles`, `y`, `split`, `seed`, `mol`, and the selected RDKit `conf_id`.

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
from tdc.benchmark_group import admet_group

from src.gnn.config import GNNConfig
from src.gnn.graph_builder import build_graph

In [ ]:
REPO_ROOT = Path.cwd().resolve().parent
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
CONFORMER_PATH = REPO_ROOT / "data" / "conformers" / "aqsoldb" / "conformers.pkl"

config = GNNConfig()
print("graph_config_id:", config.graph_config_id)
print("conformer artifact:", CONFORMER_PATH)

In [ ]:
# Load the official raw benchmark partitions without changing their rows.
group = admet_group(path=str(RAW_DATA_DIR))
benchmark = group.get("Solubility_AqSolDB")

train_val = benchmark["train_val"].copy()
test = benchmark["test"].copy()
benchmark_records = pd.concat(
    [
        train_val.assign(split="train_val"),
        test.assign(split="test"),
    ],
    ignore_index=True,
)
benchmark_records = benchmark_records.rename(
    columns={"Drug_ID": "mol_id", "Drug": "smiles", "Y": "y"}
)
benchmark_records["seed"] = None
print("loaded benchmark records:", len(benchmark_records))

In [ ]:
if not CONFORMER_PATH.is_file():
    raise FileNotFoundError(
        "Expected persisted conformers at "
        f"{CONFORMER_PATH}. Generate or restore this artifact before running T9.1."
    )

with CONFORMER_PATH.open("rb") as handle:
    conformer_records = pickle.load(handle)

if not isinstance(conformer_records, list):
    raise TypeError("conformer artifact must contain a list of records.")
required_fields = {"mol_id", "smiles", "y", "split", "seed", "mol", "conf_id"}
missing_fields = required_fields - set(conformer_records[0]) if conformer_records else required_fields
if missing_fields:
    raise ValueError(f"conformer records are missing fields: {sorted(missing_fields)}")

benchmark_ids = set(benchmark_records["mol_id"])
conformer_ids = {record["mol_id"] for record in conformer_records}
if not conformer_ids <= benchmark_ids:
    raise ValueError("conformer artifact contains IDs absent from AqSolDB.")

records = conformer_records
print("loaded conformer records:", len(records))
print("configuration and inputs loaded; ready to build an example graph.")

In [ ]:
import matplotlib.pyplot as plt

example_record = records[0]
example_graph = build_graph(
    example_record["mol"],
    conf_id=example_record["conf_id"],
    config=config,
    y=example_record["y"],
    smiles=example_record["smiles"],
    mol_id=example_record["mol_id"],
)

print("example mol_id:", example_graph.mol_id)
print("nodes:", example_graph.num_nodes, "edges:", example_graph.edge_index.shape[1])

positions = example_graph.pos.detach().cpu().numpy()
edge_index = example_graph.edge_index.detach().cpu().numpy()
fig = plt.figure(figsize=(8, 6))
axis = fig.add_subplot(111, projection="3d")
for source, target in edge_index.T:
    axis.plot(
        positions[[source, target], 0],
        positions[[source, target], 1],
        positions[[source, target], 2],
        color="0.65",
        linewidth=1.0,
    )
axis.scatter(positions[:, 0], positions[:, 1], positions[:, 2], s=45, color="tab:blue")
for node_index, (x_coord, y_coord, z_coord) in enumerate(positions):
    axis.text(x_coord, y_coord, z_coord, str(node_index), fontsize=8)
axis.set_title(f"G7 example graph — {example_graph.mol_id}")
axis.set_xlabel("x (Å)")
axis.set_ylabel("y (Å)")
axis.set_zlabel("z (Å)")
plt.tight_layout()
plt.show()